# DLM Intermediate Tutorial — Setup & Notation Review

This notebook verifies your environment and recaps the core DLM notation before the five topic notebooks.

**Prerequisites:** you have worked through the Beginner Streamlit tutorial (local level, local linear trend, seasonal) and are comfortable with the Kalman filter update equations.

## 1. Environment check

In [ ]:
import sys
from pathlib import Path

# Add project root to path if running from notebooks/intermediate/
project_root = Path().resolve().parents[1]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Engine imports — these should all succeed
from engine.models import DLMSpec, DLMSpecTV, make_local_level, make_fourier_seasonal
from engine.filter import kalman_filter, kalman_filter_discount, kalman_filter_tv
from engine.conjugate import kalman_filter_conjugate
from engine.comparison import compare_models
from engine.simulate import simulate

print("All imports OK.")
print(f"NumPy {np.__version__}, pandas {pd.__version__}")

## 2. Notation recap

The Gaussian DLM (West & Harrison notation):

$$
\begin{aligned}
y_t &= F_t \theta_t + v_t, \quad v_t \sim N(0, V_t) \quad \text{(observation equation)}\\
\theta_t &= G_t \theta_{t-1} + w_t, \quad w_t \sim N(0, W_t) \quad \text{(state equation)}
\end{aligned}
$$

| Symbol | Dimension | Meaning |
|--------|-----------|----------|
| $y_t$ | $(p,)$ | observation vector |
| $\theta_t$ | $(d,)$ | latent state vector |
| $F_t$ | $(p, d)$ | observation matrix |
| $G_t$ | $(d, d)$ | state transition matrix |
| $V_t$ | $(p, p)$ | observation noise covariance |
| $W_t$ | $(d, d)$ | state evolution noise covariance |

The **Kalman filter** produces, at each time $t$:

$$
\theta_t \mid y_{1:t} \sim N(m_t, C_t)
$$

by iterating the predict–update cycle:

$$
\underbrace{a_t = G_t m_{t-1}, \quad R_t = G_t C_{t-1} G_t' + W_t}_{\text{predict}}
\qquad
\underbrace{m_t = a_t + A_t e_t, \quad C_t = (I - A_t F_t) R_t}_{\text{update}}
$$

where $e_t = y_t - F_t a_t$ is the **innovation** and $A_t = R_t F_t' Q_t^{-1}$ is the **Kalman gain**.

## 3. Quick smoke test — local-level simulation and filter

In [ ]:
# Generate a local-level series
spec = make_local_level(V=1.0, W_level=0.1)
sim = simulate(spec, n=100, seed=0)
y = sim.y   # (100, 1)

# Run the Kalman filter
fr = kalman_filter(spec, y)

print(f"Log marginal likelihood: {fr.loglik:.2f}")
print(f"Filtered state shape: {fr.m.shape}")

In [ ]:
t = np.arange(len(y))
std = np.sqrt(fr.C[:, 0, 0])

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t, y[:, 0], 'o', ms=3, alpha=0.5, label='observations')
ax.plot(t, sim.theta_true[:, 0], 'k--', lw=1, label='true state')
ax.plot(t, fr.m[:, 0], 'C0', label='filtered mean')
ax.fill_between(t, fr.m[:, 0] - 2*std, fr.m[:, 0] + 2*std, alpha=0.2, label='±2σ')
ax.set(title='Local-level filter', xlabel='t', ylabel='y')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Intermediate topics overview

| Notebook | Topic | New engine function |
|----------|-------|--------------------|
| `01_discount_factors.ipynb` | Replace $W_t$ with a discount factor $\delta$ | `kalman_filter_discount` |
| `02_conjugate_unknown_variance.ipynb` | Estimate $V$ online with an inverse-gamma prior | `kalman_filter_conjugate` |
| `03_dynamic_regression.ipynb` | Time-varying coefficients via time-varying $F_t$ | `kalman_filter_tv`, `DLMSpecTV` |
| `04_fourier_seasonal.ipynb` | Fourier-form seasonal representation | `make_fourier_seasonal` |
| `05_model_comparison.ipynb` | Bayes factors via log marginal likelihoods | `compare_models` |

Work through them in order — each notebook assumes familiarity with the previous ones.